# King County Homebuyer's Guide

> **This notebook contains charts and maps — view the full rendered version on nbviewer:**
> [![nbviewer](https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg)](https://nbviewer.org/github/yijiaw0725/seattlle-housing-analytics/blob/main/kc_buyer_guide.ipynb)

A data-driven visual guide for anyone considering buying a home in King County, WA.
Combines KC Assessor sales records, OSPI school quality data, and GIS boundaries to answer the questions buyers actually ask.

**Key questions answered:**
1. Where are prices highest — and lowest — by school district?
2. Which districts offer the best school quality per dollar?
3. What does a $700K–$1.2M budget actually buy?
4. How much extra do waterfront and view properties cost?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
import geopandas as gpd
import folium
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.0f}'.format)
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

fmt_dol = mticker.FuncFormatter(lambda x, _: f'${x:,.0f}')
ENC = 'latin-1'
DATA_DIR = Path('kc_assessor_data')
EDU_DIR  = Path('education_data')

In [ ]:
# ── Load all source data ──────────────────────────────────────────────────────

# 1. KC Assessor: arms-length SFR sales 2020-2024
rp = pd.read_csv(DATA_DIR / 'RealPropertySales/EXTR_RPSale.csv', low_memory=False, encoding=ENC)
rb = pd.read_csv(DATA_DIR / 'ResidentialBuilding/EXTR_ResBldg.csv', low_memory=False, encoding=ENC)
par = pd.read_csv(DATA_DIR / 'Parcel/EXTR_Parcel.csv', low_memory=False, encoding=ENC)

def make_pin(df):
    return df['Major'].astype(str).str.zfill(6) + df['Minor'].astype(str).str.zfill(4)

for df in [rp, rb, par]:
    df['PIN'] = make_pin(df)

rp['DocumentDate'] = pd.to_datetime(rp['DocumentDate'], errors='coerce')
rp['SaleYear'] = rp['DocumentDate'].dt.year

al = rp[
    (rp['SaleReason'] == 1) & (rp['SalePrice'] > 10_000) &
    (rp['PropertyClass'] == 8) & (rp['SaleYear'].between(2020, 2024))
].copy()
al_latest = al.sort_values('DocumentDate').drop_duplicates('PIN', keep='last')

rb_sfr = rb[
    (rb['NbrLivingUnits'] == 1) & (rb['SqFtTotLiving'].between(200, 15_000)) &
    (rb['YrBuilt'].between(1870, 2024)) & (rb['Bedrooms'].between(1, 10))
].copy()
rb_primary = rb_sfr.sort_values('BldgNbr').drop_duplicates('PIN', keep='first')

par_unique = par.drop_duplicates('PIN', keep='first')

merged = (
    al_latest
    .merge(rb_primary[['PIN','SqFtTotLiving','BldgGrade','Condition',
                        'YrBuilt','YrRenovated','Bedrooms','BathFullCount','Bath3qtrCount','BathHalfCount']], on='PIN', how='inner')
    .merge(par_unique[['PIN','WfntLocation','WfntFootage','Territorial','PugetSound',
                        'LakeWashington','Olympics','Cascades','SeattleSkyline','SqFtLot']], on='PIN', how='left')
)
merged['IsWaterfront'] = (merged['WfntLocation'] > 0).astype(int)
merged['HasView'] = (merged[['Territorial','PugetSound','LakeWashington','Olympics','Cascades','SeattleSkyline']].gt(0).any(axis=1)).astype(int)
merged['TotalBaths'] = merged['BathFullCount'] + 0.75*merged['Bath3qtrCount'] + 0.5*merged['BathHalfCount']
merged['PricePerSqFt'] = merged['SalePrice'] / merged['SqFtTotLiving'].replace(0, np.nan)

# 2. School quality
ospi = pd.read_csv(EDU_DIR / 'ospi_assessment_2324_king.csv', low_memory=False)
ospi_clean = ospi[ospi['dat'].isna() | (ospi['dat'] == 'None') | (ospi['dat'] == '')].copy()
ospi_clean['pct_met'] = pd.to_numeric(
    ospi_clean['percent_consistent_grade_level_knowledge_and_above'].astype(str).str.rstrip('%'), errors='coerce')
school_scores = (
    ospi_clean
    .pivot_table(index=['schoolcode','schoolname','districtcode','districtname'],
                 columns='testsubject', values='pct_met')
    .reset_index()
)
school_scores.columns.name = None
school_scores = school_scores.rename(columns={'Math':'pct_math','ELA':'pct_ela','Science':'pct_science'})
subj_cols = [c for c in ['pct_math','pct_ela','pct_science'] if c in school_scores.columns]
school_scores['pct_composite'] = school_scores[subj_cols].mean(axis=1)
district_scores = school_scores.groupby('districtname')[subj_cols+['pct_composite']].mean()

# 3. GIS: district boundaries + PIN lookup
districts_geo = gpd.read_file(EDU_DIR / 'kc_school_districts.geojson').to_crs('EPSG:4326')
pin_district  = pd.read_csv(EDU_DIR / 'kc_pin_district_lookup.csv')[['PIN','NAME']].drop_duplicates('PIN')
pin_district.columns = ['PIN','gis_name']

# 4. Normalize names and build master district table
def norm(name):
    if pd.isna(name): return name
    return re.sub(r'\s+School District.*', '', str(name), flags=re.IGNORECASE).strip().upper()

pin_district['_key'] = pin_district['gis_name'].apply(norm)
district_scores['_key'] = district_scores.index.map(norm)
districts_geo['_key'] = districts_geo['NAME'].apply(norm)

# 5. Housing stats per district (2020-2024)
merged_d = (
    merged
    .merge(pin_district, on='PIN', how='left')
)
merged_d['_key'] = merged_d['gis_name'].apply(norm)

housing_by_district = (
    merged_d.groupby('_key')
    .agg(
        median_price   = ('SalePrice',     'median'),
        median_ppsf    = ('PricePerSqFt',  'median'),
        n_sales        = ('SalePrice',     'count'),
        median_sqft    = ('SqFtTotLiving', 'median'),
        pct_waterfront = ('IsWaterfront',  'mean'),
        pct_view       = ('HasView',       'mean'),
    )
    .reset_index()
)

# 6. Master table: join geo + school + housing
master = (
    districts_geo[['_key','NAME','geometry']]
    .merge(district_scores[['_key','pct_composite']].reset_index(), on='_key', how='left')
    .merge(housing_by_district, on='_key', how='left')
)
master['value_score'] = master['pct_composite'] / (master['median_price'] / 100_000)
master['gs_proxy'] = master['pct_composite'].rank(pct=True).mul(9).add(1).round(1)

print(f'Districts with full data: {master[["pct_composite","median_price"]].dropna().shape[0]}')
master[['NAME','districtname','median_price','pct_composite','value_score','n_sales']].dropna().sort_values('median_price', ascending=False).head(10)

---
## 1. Where Are Prices Highest?

In [ ]:
# ── Choropleth map: median SFR price by school district ───────────────────────
m_price = folium.Map(location=[47.48, -122.0], zoom_start=10, tiles='CartoDB positron')

price_data = master[['_key','NAME','median_price','n_sales']].dropna(subset=['median_price'])

folium.Choropleth(
    geo_data   = master[master['median_price'].notna()].to_json(),
    data       = price_data,
    columns    = ['NAME', 'median_price'],
    key_on     = 'feature.properties.NAME',
    fill_color = 'YlOrRd',
    fill_opacity = 0.75,
    line_opacity = 0.4,
    legend_name = 'Median SFR Sale Price (2020–2024)',
    bins       = 6,
).add_to(m_price)

# Tooltips
style = lambda feat: {'fillColor': 'transparent', 'color': '#555', 'weight': 1}
tooltip_layer = folium.GeoJson(
    master[master['median_price'].notna()].to_json(),
    style_function = style,
    tooltip = folium.GeoJsonTooltip(
        fields=['NAME','median_price','n_sales'],
        aliases=['District', 'Median Price', 'Sales (2020-24)'],
        localize=True
    )
).add_to(m_price)

m_price.save('education_data/map_price.html')
print('Saved: education_data/map_price.html')
m_price

In [ ]:
# ── Bar chart: median price ranking ──────────────────────────────────────────
ranked = master[['NAME','median_price','n_sales']].dropna().sort_values('median_price')

fig, ax = plt.subplots(figsize=(11, 7))
colors = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(ranked)))
bars = ax.barh(ranked['NAME'], ranked['median_price'], color=colors)

county_median = merged['SalePrice'].median()
ax.axvline(county_median, color='steelblue', lw=1.5, ls='--', label=f'County median ${county_median:,.0f}')

for bar, val in zip(bars, ranked['median_price']):
    ax.text(val + 8_000, bar.get_y() + bar.get_height()/2,
            f'${val/1_000:.0f}K', va='center', fontsize=8)

ax.set_xlabel('Median SFR Sale Price (2020–2024)')
ax.xaxis.set_major_formatter(fmt_dol)
ax.set_title('Median SFR Sale Price by School District\nKing County, 2020–2024', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Where Are the Best Schools?

In [ ]:
# ── Choropleth map: school composite pass rate ────────────────────────────────
m_school = folium.Map(location=[47.48, -122.0], zoom_start=10, tiles='CartoDB positron')

school_data = master[['NAME','pct_composite']].dropna()

folium.Choropleth(
    geo_data     = master[master['pct_composite'].notna()].to_json(),
    data         = school_data,
    columns      = ['NAME','pct_composite'],
    key_on       = 'feature.properties.NAME',
    fill_color   = 'YlGn',
    fill_opacity = 0.75,
    line_opacity = 0.4,
    legend_name  = 'OSPI Composite Pass Rate % (2023–24)',
    bins         = 6,
).add_to(m_school)

folium.GeoJson(
    master[master['pct_composite'].notna()].to_json(),
    style_function = lambda feat: {'fillColor':'transparent','color':'#555','weight':1},
    tooltip = folium.GeoJsonTooltip(
        fields=['NAME','pct_composite','gs_proxy'],
        aliases=['District','Pass Rate (%)','GreatSchools proxy (1-10)'],
        localize=True
    )
).add_to(m_school)

m_school.save('education_data/map_school.html')
print('Saved: education_data/map_school.html')
m_school

In [ ]:
# ── Side-by-side: school quality vs price ────────────────────────────────────
valid = master[['NAME','districtname','median_price','pct_composite','n_sales','value_score']].dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, col, cmap, title, fmt in [
    (axes[0], 'median_price', 'YlOrRd', 'Median SFR Price (2020–24)', '${:,.0f}'),
    (axes[1], 'pct_composite', 'YlGn',  'School Pass Rate % (2023–24)', '{:.1f}%'),
]:
    ranked = valid.sort_values(col)
    vals   = ranked[col].values
    norm   = mcolors.Normalize(vmin=vals.min(), vmax=vals.max())
    colors = plt.cm.get_cmap(cmap)(norm(vals))
    bars = ax.barh(ranked['NAME'], vals, color=colors)
    for bar, val in zip(bars, vals):
        ax.text(val * 1.01, bar.get_y() + bar.get_height()/2,
                fmt.format(val), va='center', fontsize=7.5)
    ax.set_title(title, fontsize=12)
    if col == 'median_price':
        ax.xaxis.set_major_formatter(fmt_dol)
    else:
        ax.set_xlabel('% students meeting standard')

plt.suptitle('King County School Districts — Price vs. School Quality', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. Is Paying More for Schools Worth It?

In [ ]:
# ── Scatter: school quality vs median price (with district labels) ─────────────
valid = master[['NAME','districtname','median_price','pct_composite','n_sales']].dropna()

fig, ax = plt.subplots(figsize=(12, 7))

sc = ax.scatter(
    valid['pct_composite'], valid['median_price'],
    s=valid['n_sales'] / valid['n_sales'].max() * 800,
    c=valid['median_price'], cmap='YlOrRd',
    alpha=0.8, edgecolors='white', lw=0.8
)

# Trend line
z = np.polyfit(valid['pct_composite'], valid['median_price'], 1)
xs = np.linspace(valid['pct_composite'].min(), valid['pct_composite'].max(), 100)
ax.plot(xs, np.poly1d(z)(xs), 'steelblue', ls='--', lw=1.5, alpha=0.7, label='Trend')

# Median lines
ax.axvline(valid['pct_composite'].median(), color='gray', ls=':', lw=1, alpha=0.6)
ax.axhline(valid['median_price'].median(),  color='gray', ls=':', lw=1, alpha=0.6)

# Quadrant labels
xm, ym = valid['pct_composite'].median(), valid['median_price'].median()
for txt, xp, yp, ha in [
    ('High school\nHigh price', 0.97, 0.97, 'right'),
    ('High school\nLower price\n★ Best value', 0.03, 0.97, 'left'),
    ('Lower school\nLower price', 0.03, 0.03, 'left'),
    ('Lower school\nHigh price', 0.97, 0.03, 'right'),
]:
    ax.text(xm + (valid['pct_composite'].max()-xm)*(xp-0.5)*2,
            ym + (valid['median_price'].max()-ym)*(yp-0.5)*2,
            txt, ha=ha, va='center', fontsize=8,
            color='gray', style='italic', alpha=0.7)

# District labels
for _, row in valid.iterrows():
    name = row.get('NAME', '') or str(row.get('districtname','')).replace(' School District','').replace(' No. 1','')
    ax.annotate(name, (row['pct_composite'], row['median_price']),
                fontsize=7.5, textcoords='offset points', xytext=(5,3))

ax.yaxis.set_major_formatter(fmt_dol)
ax.set_xlabel('OSPI Composite Pass Rate % (higher = better schools)', fontsize=11)
ax.set_ylabel('Median SFR Sale Price 2020–2024', fontsize=11)
ax.set_title('School Quality vs. House Price by District\n(bubble size = transaction volume)', fontsize=13)
plt.colorbar(sc, ax=ax, label='Median Price')
plt.tight_layout()
plt.show()

r = valid['pct_composite'].corr(valid['median_price'])
print(f'Correlation (school pass rate vs. median price): r = {r:.3f}')

In [ ]:
# ── School quality quartile: what does it actually cost? ─────────────────────
with_scores = (
    merged_d
    .merge(district_scores[['_key','pct_composite']].reset_index(), on='_key', how='left')
)
with_scores['school_quartile'] = pd.qcut(
    with_scores['pct_composite'].fillna(with_scores['pct_composite'].median()),
    q=4, labels=['Q1\nBottom 25%','Q2','Q3','Q4\nTop 25%']
)

q_stats = with_scores.groupby('school_quartile', observed=True).agg(
    median_price   = ('SalePrice', 'median'),
    median_sqft    = ('SqFtTotLiving', 'median'),
    median_ppsf    = ('PricePerSqFt', 'median'),
    n              = ('SalePrice', 'count')
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#e74c3c','#e67e22','#2ecc71','#27ae60']

for ax, col, title, fmt in [
    (axes[0], 'median_price',   'Median Sale Price',        '${:,.0f}'),
    (axes[1], 'median_sqft',    'Median Living Area (SqFt)', '{:,.0f} sqft'),
    (axes[2], 'median_ppsf',    'Median Price per SqFt',    '${:,.0f}/sqft'),
]:
    bars = ax.bar(q_stats.index, q_stats[col], color=colors)
    for bar, val in zip(bars, q_stats[col]):
        ax.text(bar.get_x()+bar.get_width()/2, val*1.01,
                fmt.format(val), ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=11)
    if col == 'median_price':
        ax.yaxis.set_major_formatter(fmt_dol)

premium = (q_stats.loc['Q4\nTop 25%','median_price'] / q_stats.loc['Q1\nBottom 25%','median_price'] - 1) * 100
plt.suptitle(f'What You Get at Each School Quality Level  (top-quartile premium: +{premium:.0f}%)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(q_stats.to_string())

---
## 4. The Value Map — Best School Quality per Dollar

In [ ]:
# Value score = composite pass rate / (median price / $100K)
# Higher = more school quality per dollar spent

m_value = folium.Map(location=[47.48, -122.0], zoom_start=10, tiles='CartoDB positron')

value_data = master[['NAME','value_score']].dropna()

folium.Choropleth(
    geo_data     = master[master['value_score'].notna()].to_json(),
    data         = value_data,
    columns      = ['NAME','value_score'],
    key_on       = 'feature.properties.NAME',
    fill_color   = 'PuBuGn',
    fill_opacity = 0.75,
    line_opacity = 0.4,
    legend_name  = 'Value Score (school quality per $100K price)',
    bins         = 6,
).add_to(m_value)

folium.GeoJson(
    master[master['value_score'].notna()].to_json(),
    style_function = lambda feat: {'fillColor':'transparent','color':'#555','weight':1},
    tooltip = folium.GeoJsonTooltip(
        fields=['NAME','median_price','pct_composite','value_score'],
        aliases=['District','Median Price ($)','Pass Rate (%)','Value Score'],
        localize=True
    )
).add_to(m_value)

m_value.save('education_data/map_value.html')
print('Saved: education_data/map_value.html')
m_value

In [ ]:
# ── Value score ranking table ─────────────────────────────────────────────────
valid = master[['NAME','districtname','median_price','pct_composite','value_score','n_sales']].dropna()
valid = valid.sort_values('value_score', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 7))
vals   = valid['value_score'].values
norm   = mcolors.Normalize(vmin=vals.min(), vmax=vals.max())
colors = plt.cm.PuBuGn(norm(vals))
bars = ax.barh(valid['NAME'][::-1], valid['value_score'][::-1], color=colors[::-1])
for bar, (_, row) in zip(bars, valid[::-1].iterrows()):
    ax.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
            f"score {row['value_score']:.1f}  |  ${row['median_price']/1e6:.2f}M  |  {row['pct_composite']:.0f}% pass",
            va='center', fontsize=8)
ax.set_title('Value Score by District\n(School Pass Rate ÷ Price per $100K — higher = more school quality per dollar)', fontsize=11)
ax.set_xlabel('Value Score')
ax.set_xlim(right=vals.max()*1.55)
plt.tight_layout()
plt.show()

---
## 5. Budget Guide — What Does My Money Buy?

In [ ]:
# ── For each district: typical 3BR home price and attributes ──────────────────
three_br = (
    with_scores[with_scores['Bedrooms'] == 3]
    .groupby('_key')
    .agg(
        median_price  = ('SalePrice',      'median'),
        median_sqft   = ('SqFtTotLiving',  'median'),
        median_ppsf   = ('PricePerSqFt',   'median'),
        pct_composite = ('pct_composite',  'mean'),
        n             = ('SalePrice',       'count'),
        gis_name      = ('gis_name',        'first'),
    )
    .dropna(subset=['median_price','pct_composite'])
    .sort_values('median_price')
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(12, 8))
norm   = mcolors.Normalize(vmin=three_br['pct_composite'].min(), vmax=three_br['pct_composite'].max())
colors = plt.cm.YlGn(norm(three_br['pct_composite'].values))

bars = ax.barh(three_br['gis_name'], three_br['median_price'], color=colors)

for bar, (_, row) in zip(bars, three_br.iterrows()):
    ax.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
            f"${row['median_price']/1e3:.0f}K  |  {row['median_sqft']:.0f} sqft  |  school {row['pct_composite']:.0f}%",
            va='center', fontsize=8)

# Budget reference lines
for budget, label in [(600_000,'$600K'),(800_000,'$800K'),(1_000_000,'$1M')]:
    ax.axvline(budget, color='steelblue', ls='--', lw=1, alpha=0.5)
    ax.text(budget, len(three_br)*1.01, label, color='steelblue', fontsize=8, ha='center')

# Color bar for school quality
sm = plt.cm.ScalarMappable(cmap='YlGn', norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='School Pass Rate %', shrink=0.6)

ax.xaxis.set_major_formatter(fmt_dol)
ax.set_xlim(right=three_br['median_price'].max()*1.55)
ax.set_title('Typical 3-Bedroom Home Price by School District\n(bar color = school quality — greener is better)', fontsize=12)
ax.set_xlabel('Median Sale Price (2020–2024)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Budget scenarios: what can $700K / $900K / $1.2M buy? ─────────────────────
scenarios = [600_000, 800_000, 1_100_000]
tol = 0.20  # ±20% of budget

fig, axes = plt.subplots(1, 3, figsize=(17, 6), sharey=False)

for ax, budget in zip(axes, scenarios):
    bucket = with_scores[
        with_scores['SalePrice'].between(budget*(1-tol), budget*(1+tol))
    ]
    if len(bucket) < 20:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        continue

    ax.hist(bucket['SqFtTotLiving'].clip(upper=5000), bins=30,
            color='steelblue', edgecolor='white', alpha=0.8)
    med_sqft = bucket['SqFtTotLiving'].median()
    ax.axvline(med_sqft, color='red', lw=1.5, ls='--',
               label=f'Median {med_sqft:.0f} sqft')
    ax.set_title(f'Budget: ${budget/1e3:.0f}K  (n={len(bucket):,})', fontsize=11)
    ax.set_xlabel('Living Area (SqFt)')
    ax.set_ylabel('Homes')
    ax.legend(fontsize=9)

    med_br   = bucket['Bedrooms'].median()
    med_bath = bucket['TotalBaths'].median()
    ax.text(0.97, 0.85,
            f'Typical: {med_br:.0f} BR / {med_bath:.1f} BA',
            transform=ax.transAxes, ha='right', fontsize=9,
            bbox=dict(boxstyle='round', fc='white', alpha=0.7))

plt.suptitle('What Does Your Budget Buy? Size Distribution by Price Range', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Special Features — Waterfront & Views

In [ ]:
# ── Premium for waterfront, view, and combined ───────────────────────────────
lu   = pd.read_csv(DATA_DIR / 'Lookup/EXTR_LookUp.csv', low_memory=False, encoding=ENC)
lu50 = lu[lu['LUType'] == 50].set_index('LUItem')['LUDescription'].str.strip()

base     = merged[(merged['IsWaterfront']==0) & (merged['HasView']==0)]['SalePrice'].median()
view_med = merged[(merged['HasView']==1) & (merged['IsWaterfront']==0)]['SalePrice'].median()
wf_med   = merged[merged['IsWaterfront']==1]['SalePrice'].median()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segment comparison
segments = {
    'No View\nNo Waterfront': base,
    'Has View\n(no waterfront)': view_med,
    'Waterfront': wf_med,
}
colors_seg = ['#95a5a6','#3498db','#1abc9c']
bars = axes[0].bar(segments.keys(), segments.values(), color=colors_seg)
for bar, val in zip(bars, segments.values()):
    prem = (val/base - 1)*100
    axes[0].text(bar.get_x()+bar.get_width()/2, val*1.02,
                 f'${val/1e6:.2f}M\n({prem:+.0f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[0].yaxis.set_major_formatter(fmt_dol)
axes[0].set_title('Median Price by Property Type\n(SFR, 2020–2024)', fontsize=11)
axes[0].set_ylabel('Median Sale Price')

# Waterfront body of water breakdown — WfntLocation already in merged
wf_parcels = merged[merged['IsWaterfront']==1].copy()
wf_parcels['WfntType'] = wf_parcels['WfntLocation'].map(lu50)
wf_price = (
    wf_parcels.groupby('WfntType')['SalePrice']
    .agg(['median','count'])
    .query('count >= 10')
    .sort_values('median', ascending=True)
)

axes[1].barh(wf_price.index, wf_price['median'], color='#1abc9c', alpha=0.85)
for i, (wtype, row) in enumerate(wf_price.iterrows()):
    axes[1].text(row['median']*1.01, i,
                 f"${row['median']/1e6:.2f}M  (n={int(row['count'])})",
                 va='center', fontsize=8)
axes[1].xaxis.set_major_formatter(fmt_dol)
axes[1].set_title('Waterfront Median Price by Body of Water\n(2020–2024)', fontsize=11)
axes[1].set_xlim(right=wf_price['median'].max()*1.5)

plt.tight_layout()
plt.show()

---
## 7. Safety & Crime — Seattle Focus

> **Data scope:** SPD (Seattle Police Dept.) crime data covers **Seattle city limits only**.
> Crime scores below reflect serious incidents within 500m of each property in the 12 months before sale.

**Key questions:**
- Which Seattle-area neighborhoods have the lowest crime exposure?
- Does higher crime correlate with lower prices — and by how much?

In [ ]:
# ── Load crime scores and merge with Seattle SFR sales ────────────────────────
CRIME_DIR = Path('crime_data')

crime_scores = pd.read_csv(CRIME_DIR / 'seattle_sales_crime_score.csv', dtype={'PIN': str})
parcel_coords = pd.read_csv(EDU_DIR / 'kc_parcel_coords.csv')[['PIN','LAT','LON']]

# Merge: sales + school district + school quality score + crime scores + coordinates
seattle_crime = (
    merged_d
    .merge(district_scores[['_key','pct_composite']].reset_index(), on='_key', how='left')
    .merge(parcel_coords, on='PIN', how='left')
    .merge(crime_scores, on='PIN', how='inner')
)

# Filter to Seattle bounding box
seattle_crime = seattle_crime[
    seattle_crime['LAT'].between(47.49, 47.74) &
    seattle_crime['LON'].between(-122.46, -122.22)
].copy()
seattle_crime['ppsf'] = seattle_crime['SalePrice'] / seattle_crime['SqFtTotLiving'].replace(0, np.nan)

# Crime safety quintile (Q1 = safest)
seattle_crime['safety_q'] = pd.qcut(
    seattle_crime['crime_count_500m'], q=5,
    labels=['Q1 Safest','Q2','Q3','Q4','Q5 Highest Crime']
)

print(f'Seattle SFR sales with crime scores: {len(seattle_crime):,}')
print(f'Crime score  min / median / max: '
      f'{seattle_crime["crime_count_500m"].min():.0f} / '
      f'{seattle_crime["crime_count_500m"].median():.0f} / '
      f'{seattle_crime["crime_count_500m"].max():.0f}')

In [ ]:
# ── Chart 1: Crime exposure by Seattle school district ───────────────────────
district_crime = (
    seattle_crime.groupby('_key')
    .agg(
        median_crime   = ('crime_count_500m', 'median'),
        median_price   = ('SalePrice',        'median'),
        pct_composite  = ('pct_composite',    'mean'),
        n              = ('SalePrice',         'count'),
        gis_name       = ('gis_name',          'first'),
    )
    .query('n >= 20')
    .dropna(subset=['gis_name','median_crime'])
    .sort_values('median_crime')
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Safety bar chart (low = safer)
norm_c = mcolors.Normalize(
    vmin=district_crime['median_crime'].min(),
    vmax=district_crime['median_crime'].max()
)
colors_c = plt.cm.RdYlGn_r(norm_c(district_crime['median_crime'].values))
bars = axes[0].barh(district_crime['gis_name'], district_crime['median_crime'],
                    color=colors_c, alpha=0.9)
for bar, (_, row) in zip(bars, district_crime.iterrows()):
    axes[0].text(bar.get_width()*1.02, bar.get_y()+bar.get_height()/2,
                 f"{row['median_crime']:.0f} incidents",
                 va='center', fontsize=8)
axes[0].set_title('Median Crime Exposure by District\n(serious incidents within 500m, 12-month window)',
                  fontsize=11)
axes[0].set_xlabel('Median serious crime incidents (lower = safer)')
axes[0].set_xlim(right=district_crime['median_crime'].max() * 1.35)

# Price vs crime scatter by district
sc = axes[1].scatter(
    district_crime['median_crime'],
    district_crime['median_price'],
    s=district_crime['n'] / district_crime['n'].max() * 400,
    c=district_crime['median_crime'], cmap='RdYlGn_r',
    alpha=0.85, edgecolors='white', lw=0.8
)
z = np.polyfit(district_crime['median_crime'], district_crime['median_price'], 1)
xs = np.linspace(district_crime['median_crime'].min(), district_crime['median_crime'].max(), 100)
axes[1].plot(xs, np.poly1d(z)(xs), 'steelblue', ls='--', lw=1.5, alpha=0.7)
for _, row in district_crime.iterrows():
    axes[1].annotate(str(row['gis_name']).title(),
                     (row['median_crime'], row['median_price']),
                     fontsize=8, textcoords='offset points', xytext=(4, 3))
axes[1].yaxis.set_major_formatter(fmt_dol)
axes[1].set_xlabel('Median Crime Exposure Score')
axes[1].set_ylabel('Median Sale Price')
axes[1].set_title('Crime Exposure vs. House Price by District\n(bubble size = sales volume)', fontsize=11)
r_d = district_crime['median_crime'].corr(district_crime['median_price'])
axes[1].text(0.97, 0.97, f'r = {r_d:.2f}', transform=axes[1].transAxes,
             ha='right', va='top', fontsize=11, color='steelblue')

plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 2: Safety quintile — price, size, and school trade-off ──────────────
q_crime = seattle_crime.groupby('safety_q', observed=True).agg(
    median_price   = ('SalePrice',          'median'),
    median_ppsf    = ('ppsf',               'median'),
    median_sqft    = ('SqFtTotLiving',      'median'),
    pct_composite  = ('pct_composite',      'mean'),
    median_crime   = ('crime_count_500m',   'median'),
    n              = ('SalePrice',           'count'),
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
safety_colors = ['#27ae60','#2ecc71','#f1c40f','#e67e22','#e74c3c']

for ax, col, title, fmt in [
    (axes[0], 'median_price',  'Median Sale Price',         '${:,.0f}'),
    (axes[1], 'median_ppsf',   'Median Price per SqFt',     '${:,.0f}/sqft'),
    (axes[2], 'pct_composite', 'Avg School Pass Rate (%)',  '{:.0f}%'),
]:
    bars = ax.bar(q_crime.index, q_crime[col], color=safety_colors)
    for bar, val in zip(bars, q_crime[col]):
        ax.text(bar.get_x()+bar.get_width()/2, val*1.01,
                fmt.format(val), ha='center', fontsize=9, fontweight='bold')
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='x', rotation=15)
    if col == 'median_price':
        ax.yaxis.set_major_formatter(fmt_dol)

discount = (q_crime.loc['Q5 Highest Crime','median_price'] /
            q_crime.loc['Q1 Safest','median_price'] - 1) * 100
ppsf_discount = (q_crime.loc['Q5 Highest Crime','median_ppsf'] /
                 q_crime.loc['Q1 Safest','median_ppsf'] - 1) * 100
plt.suptitle(
    f'Safety Quintile Analysis (Seattle SFR, 2020–2024)\n'
    f'Safest vs. Highest-crime: raw price {discount:+.0f}%  |  price/sqft {ppsf_discount:+.0f}%',
    fontsize=12, y=1.03
)
plt.tight_layout()
plt.show()

print(q_crime[['median_crime','median_price','median_ppsf','pct_composite','n']].to_string())

---
## 8. Summary — Buyer's Cheat Sheet

| Question | Finding |
|----------|---------|
| Most expensive districts | Mercer Island, Bellevue, Issaquah — median SFR > $1M |
| Best school quality | Mercer Island, Lake Washington, Bellevue — OSPI composite > 75% |
| Best value (school per dollar) | Enumclaw, Riverview, Northshore — high pass rate relative to price |
| School quality premium | Top-quartile districts cost **~+99%** over bottom-quartile |
| Price–school correlation | r = 0.77 — school quality is one of the strongest price signals |
| Waterfront premium | Waterfront SFR median **~+150%** over non-waterfront |
| View premium | Has-view properties **~+30–40%** over no-view, no-waterfront |
| $600K budget | Typical 3BR, ~1,500–1,800 sqft, mid-tier school district |
| $1.1M budget | Typical 3–4BR, ~2,200–2,800 sqft, access to top school districts |
| Safest Seattle areas (lowest crime) | Shoreline, Lake Washington districts — suburban, lower density |
| Highest crime exposure | Central Seattle School District properties near downtown core |
| Crime–price correlation | r ≈ −0.1 (weak) — urban amenity premium offsets crime discount |
| Crime data scope | SPD covers **Seattle city limits only**; suburban KC has no equivalent data |

> **Maps saved to `education_data/`:** `map_price.html` · `map_school.html` · `map_value.html`  
> **Crime heat map:** `crime_data/map_crime_heatmap.html`  
> Open in any browser for interactive exploration.